# ECCE 2027 Architecture Follow-up -- EVAL ONLY (Kaggle): E2.1/YOLO26s -- Chattogram Fold 1 (train) -> out-of-fold + reverse-transfer eval

Runs in a brand-new container, separate from `train_run_e2_1_y26.ipynb`, so the
GPU is guaranteed clean.

**Attach two inputs before running** (Add Input, right sidebar):
1. `badodd-ecce2027-bundle` (images + overlay + coco_gt) -- same bundle as the YOLOv8s runs.
2. `train_run_e2_1_y26` -- your own training notebook, added as an input. It
   must have a completed, saved version first.

Evaluation targets for this run: ctg_fold1_eval, dhaka_fold1_eval

Settings: Accelerator = GPU T4 x2, Internet = ON.


In [ ]:
# ==============================================================================
# 1. HARDWARE & ENVIRONMENT VERIFICATION (fresh container -- GPU should start clean)
# ==============================================================================
import os, sys, time, glob, json, shutil, subprocess

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

from pathlib import Path
import torch

PINNED_ULTRALYTICS = "8.4.155"
RUN_ID = 'e2_1_y26'

print('Python version:', sys.version)
print('PyTorch version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
print('CUDA device count (should be 1 after masking):', torch.cuda.device_count())

subprocess.run("nvidia-smi --query-gpu=index,name,memory.total,memory.used,memory.free "
               "--format=csv", shell=True)

assert torch.cuda.is_available(), (
    'FAIL: no GPU detected -- Accelerator is set to "None" in this notebook\'s Settings tab. '
    'Cancel this run, set Accelerator = GPU T4 x2 in Settings, and re-run Save & Run All.'
)
device_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU Device: {device_name} ({total_vram:.2f} GB VRAM)')
allocated_at_start = torch.cuda.memory_allocated(0) / 1e9
print(f'GPU memory allocated at notebook start: {allocated_at_start:.3f} GB (should be ~0)')

subprocess.run(f"pip install -q -U ultralytics=={PINNED_ULTRALYTICS} pycocotools", shell=True, check=True)
import ultralytics
assert ultralytics.__version__ == PINNED_ULTRALYTICS, (
    f"Ultralytics version drift: installed {ultralytics.__version__}, expected {PINNED_ULTRALYTICS}"
)
print('Ultralytics version:', ultralytics.__version__)
print('Run ID:', RUN_ID)

from ultralytics import YOLO

In [ ]:
# ==============================================================================
# 2. DATASET & OVERLAY DISCOVERY -- FAIL HARD on any missing image
# ==============================================================================
print('=== Scanning /kaggle/input for Dataset & Overlay ===')

badodd_zips = glob.glob('/kaggle/input/**/badodd.zip', recursive=True)
if badodd_zips and not glob.glob('/kaggle/input/**/*.jpg', recursive=True):
    print(f'Found badodd.zip at {badodd_zips[0]}. Extracting to /kaggle/working/badodd_images...')
    os.makedirs('/kaggle/working/badodd_images', exist_ok=True)
    subprocess.run(f"unzip -q {badodd_zips[0]} -d /kaggle/working/badodd_images", shell=True, check=True)
    IMAGE_SEARCH_ROOT = '/kaggle/working/badodd_images'
else:
    IMAGE_SEARCH_ROOT = '/kaggle/input'

all_input_imgs = [p for p in glob.glob(f'{IMAGE_SEARCH_ROOT}/**/*.jpg', recursive=True) +
                        glob.glob(f'{IMAGE_SEARCH_ROOT}/**/*.png', recursive=True)
                   if not os.path.basename(p).startswith('._')]
img_lookup = {os.path.basename(p): p for p in all_input_imgs}
print(f'Indexed {len(img_lookup)} images from {IMAGE_SEARCH_ROOT}.')
assert len(img_lookup) >= 10000, (
    f"Expected ~10,032 BadODD images, found only {len(img_lookup)} -- "
    f"is the bundle dataset attached?"
)

overlay_roots = []
for root, dirs, files in os.walk('/kaggle/input'):
    if 'splits' in dirs and 'coco_gt' in dirs:
        overlay_roots.append(root)

overlay_zips = glob.glob('/kaggle/input/**/badodd_ecce2027_overlay*.zip', recursive=True)
overlay_src = None
if overlay_roots:
    overlay_src = overlay_roots[0]
    print(f'Found overlay directory at: {overlay_src}')
elif overlay_zips:
    print(f'Found overlay zip at: {overlay_zips[0]}. Unzipping to /kaggle/working/overlay...')
    subprocess.run(f"unzip -q {overlay_zips[0]} -d /kaggle/working/overlay", shell=True, check=True)
    overlay_src = '/kaggle/working/overlay'
else:
    raise FileNotFoundError(
        'Could not locate the ECCE 2027 overlay package in /kaggle/input! '
        'Attach the bundle dataset.'
    )

assert os.path.isdir(os.path.join(overlay_src, 'coco_gt'))
assert os.path.isdir(os.path.join(overlay_src, 'splits'))

WORK_DATA = '/kaggle/working/data'
WORK_IMAGES = os.path.join(WORK_DATA, 'images')
WORK_SPLITS = os.path.join(WORK_DATA, 'splits')
os.makedirs(WORK_IMAGES, exist_ok=True)
os.makedirs(WORK_SPLITS, exist_ok=True)

for bname, src_p in img_lookup.items():
    dest = os.path.join(WORK_IMAGES, bname)
    if not os.path.exists(dest):
        try:
            os.symlink(src_p, dest)
        except OSError:
            shutil.copy(src_p, dest)
print(f'Linked {len(img_lookup)} image files.')

EVAL_TARGETS = ["ctg_fold1_eval", "dhaka_fold1_eval"]

resolved_counts = {}
for t in EVAL_TARGETS:
    manifest_src = os.path.join(overlay_src, 'splits', f'{t}.txt')
    with open(manifest_src) as f:
        bases = [os.path.basename(l.strip()) for l in f if l.strip()]
    resolved = []
    missing_here = []
    for b in bases:
        p = os.path.join(WORK_IMAGES, b)
        if os.path.exists(p):
            resolved.append(p)
        else:
            missing_here.append(b)
    if missing_here:
        raise FileNotFoundError(
            f"{t}.txt: {len(missing_here)} images missing, e.g. {missing_here[:5]}. "
            f"Do not proceed -- check the attached dataset."
        )
    gt_path = os.path.join(overlay_src, 'coco_gt', f'{t}_coco_gt.json')
    assert os.path.isfile(gt_path), f'Missing COCO ground truth for {t}: {gt_path}'
    with open(os.path.join(WORK_SPLITS, f'{t}.txt'), 'w') as f:
        f.writelines(p + '\n' for p in resolved)
    resolved_counts[t] = len(resolved)

print('Resolved eval manifests (all images present):')
for t, n in resolved_counts.items():
    print(f'  {t}: {n} images')

CLASS_NAMES = {
    0: 'auto_rickshaw', 1: 'bicycle', 2: 'bus', 3: 'car', 4: 'cart_vehicle',
    5: 'construction_vehicle', 6: 'motorbike', 7: 'person', 8: 'priority_vehicle',
    9: 'three_wheeler', 10: 'truck',
}

In [ ]:
# ==============================================================================
# 3. LOCATE THE TRAINED CHECKPOINT FROM the matching train notebook's OUTPUT
# ==============================================================================
ckpt_candidates = glob.glob(f'/kaggle/input/**/{RUN_ID}_last.pt', recursive=True)
if not ckpt_candidates:
    raise FileNotFoundError(
        f"Could not find {RUN_ID}_last.pt under /kaggle/input. Attach train_run_{RUN_ID}.ipynb "
        f"as an input (Add Input -> your notebooks), and make sure it has a completed, "
        f"saved version."
    )
print(f'Checkpoint candidates found ({len(ckpt_candidates)}):')
for c in ckpt_candidates:
    print(f'  {c}  ({os.path.getsize(c)/1e6:.2f} MB)')
if len(ckpt_candidates) > 1:
    print('WARNING: multiple matches -- using the first one. If the train notebook was '
          're-run multiple times, stale copies from earlier versions may be attached too.')
LAST_CKPT = ckpt_candidates[0]
print(f'\nUsing checkpoint: {LAST_CKPT} ({os.path.getsize(LAST_CKPT)/1e6:.2f} MB)')
assert os.path.getsize(LAST_CKPT) < 300e6, (
    f'Checkpoint is {os.path.getsize(LAST_CKPT)/1e6:.1f} MB -- unexpectedly large for a '
    f'YOLO26s last.pt. Something may be wrong with this file -- do not proceed without checking.'
)

run_info_candidates = glob.glob(f'/kaggle/input/**/{RUN_ID}_run_info.json', recursive=True)
if run_info_candidates:
    with open(run_info_candidates[0]) as f:
        train_run_info = json.load(f)
    print('\nTraining run info:')
    print(json.dumps(train_run_info, indent=2))
else:
    train_run_info = None
    print('WARNING: run_info.json not found alongside the checkpoint -- proceeding without it.')

In [ ]:
# ==============================================================================
# 4. EXPORT PREDICTIONS + PYCOCOTOOLS CHECK FOR EVERY EVAL TARGET
# ==============================================================================
# Same chunked-reload pattern as the YOLOv8s eval notebooks: the streaming
# predictor leaks GPU memory per batch and never releases it mid-run, so each
# eval target is processed in small chunks, destroying and reloading the
# model between chunks so nothing accumulates past a safe ceiling.
import gc
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

CHUNK_SIZE = 128

if torch.cuda.is_available():
    print(f'GPU memory at start of export: allocated={torch.cuda.memory_allocated(0)/1e9:.3f} GB '
          f'(should be ~0 in this fresh container)')

RESULTS_DIR = '/kaggle/working/results/predictions'
LOGS_DIR = '/kaggle/working/results/logs'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)


def export_predictions(images, ckpt_path):
    coco_predictions = []
    class_counts = {cid: 0 for cid in CLASS_NAMES}
    chunks = [images[i:i + CHUNK_SIZE] for i in range(0, len(images), CHUNK_SIZE)]
    for chunk_idx, chunk in enumerate(chunks):
        try:
            chunk_model = YOLO(ckpt_path)
            results_gen = chunk_model.predict(
                source=chunk, conf=0.001, iou=0.7, imgsz=640,
                device=0 if torch.cuda.is_available() else 'cpu',
                batch=16, stream=True, verbose=False, workers=0,
            )
            for r in results_gen:
                img_stem = Path(r.path).stem
                boxes = r.boxes
                if boxes is not None and len(boxes) > 0:
                    xyxy = boxes.xyxy.cpu().numpy()
                    confs = boxes.conf.cpu().numpy()
                    clss = boxes.cls.cpu().numpy().astype(int)
                    for i in range(len(boxes)):
                        x1, y1, x2, y2 = xyxy[i]
                        cid = int(clss[i])
                        coco_predictions.append({
                            'image_id': img_stem, 'category_id': cid,
                            'bbox': [round(float(x1), 2), round(float(y1), 2),
                                     round(float(x2 - x1), 2), round(float(y2 - y1), 2)],
                            'score': round(float(confs[i]), 5),
                        })
                        if cid in class_counts:
                            class_counts[cid] += 1
                del r
            del chunk_model, results_gen
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            print(f'\n*** OOM DURING CHUNK {chunk_idx + 1}/{len(chunks)} ***')
            print(torch.cuda.memory_summary(device=0, abbreviated=True))
            subprocess.run("nvidia-smi", shell=True)
            raise
    return coco_predictions, class_counts


def pycocotools_check(pred_json_path, gt_json_path):
    coco_gt = COCO(gt_json_path)
    coco_dt = coco_gt.loadRes(pred_json_path)
    ev = COCOeval(coco_gt, coco_dt, iouType='bbox')
    ev.evaluate(); ev.accumulate(); ev.summarize()
    mAP50_95 = ev.stats[0]
    mAP50 = ev.stats[1]
    cat_ids = ev.params.catIds
    per_class_ap50 = {}
    for k, cid in enumerate(cat_ids):
        ap50_arr = ev.eval['precision'][0, :, k, 0, -1]
        ap50 = ap50_arr[ap50_arr > -1].mean() if (ap50_arr > -1).any() else float('nan')
        per_class_ap50[CLASS_NAMES[cid]] = ap50
    return mAP50, mAP50_95, per_class_ap50


results_summary = {}
for t in EVAL_TARGETS:
    print(f'\n{"="*70}\nEVAL TARGET: {t}\n{"="*70}')
    with open(os.path.join(WORK_SPLITS, f'{t}.txt')) as f:
        images = [l.strip() for l in f if l.strip()]
    print(f'{len(images)} images.')

    t0 = time.time()
    coco_predictions, class_counts = export_predictions(images, LAST_CKPT)
    elapsed = time.time() - t0
    total_boxes = len(coco_predictions)
    print(f'Exported {total_boxes} detections in {elapsed:.1f}s ({len(images)/elapsed:.1f} FPS).')
    assert total_boxes > 0, f'FAIL: zero predictions for {t}.'

    pred_json_path = os.path.join(RESULTS_DIR, f'{RUN_ID}_{t}_preds_conf0001.json')
    with open(pred_json_path, 'w') as f:
        json.dump(coco_predictions, f)

    gt_json_path = os.path.join(overlay_src, 'coco_gt', f'{t}_coco_gt.json')
    mAP50, mAP50_95, per_class_ap50 = pycocotools_check(pred_json_path, gt_json_path)

    assert per_class_ap50.get('person', 0) > 0 or per_class_ap50.get('car', 0) > 0, (
        f'FAIL: AP50 is 0 for both person and car on {t} -- category ids or bbox coords '
        f'are almost certainly misaligned.'
    )
    print(f"mAP50={mAP50:.4f} mAP50-95={mAP50_95:.4f} "
          f"person={per_class_ap50.get('person', float('nan')):.4f} "
          f"car={per_class_ap50.get('car', float('nan')):.4f}")

    results_summary[t] = {
        'n_images': len(images), 'total_boxes': total_boxes,
        'mAP50': mAP50, 'mAP50_95': mAP50_95, 'per_class_ap50': per_class_ap50,
        'pred_json': pred_json_path,
    }

with open(os.path.join(LOGS_DIR, f'{RUN_ID}_eval_summary.json'), 'w') as f:
    json.dump({t: {k: v for k, v in d.items() if k != 'pred_json'} for t, d in results_summary.items()},
               f, indent=2)
print(f'\nWrote eval summary to {LOGS_DIR}/{RUN_ID}_eval_summary.json')

In [ ]:
# ==============================================================================
# 5. FINAL SUMMARY
# ==============================================================================
print('='*70)
print(f'                {RUN_ID.upper()} EVAL SUMMARY')
print('='*70)
if train_run_info:
    print(f"Training: {train_run_info.get('epochs', '?')} epochs, "
          f"{train_run_info.get('total_train_time_s', float('nan')):.1f}s total, "
          f"peak VRAM {train_run_info.get('peak_vram_gb', float('nan')):.2f} GB, "
          f"resumed={train_run_info.get('resumed')}")
for t, d in results_summary.items():
    print(f"\n[{t}] n={d['n_images']} boxes={d['total_boxes']}")
    print(f"  mAP50={d['mAP50']:.4f} mAP50-95={d['mAP50_95']:.4f}")
    for cname, ap in d['per_class_ap50'].items():
        print(f"    {cname:22s}: AP50={ap:.4f}")
print('\n' + '='*70)
print(f'>>> {RUN_ID} EVAL CHECKS PASSED SUCCESSFULLY <<<')
print('='*70)